# Session 2: Backpropagation & full training loop

In [ ]:
# Let's go ahead and do all of the imports we need now
import torch    
from torchvision.transforms import ToTensor
from torchvision.datasets import MNIST
from tqdm import tqdm

In [5]:
class SimpleModel(torch.nn.Module):
  def __init__(self, input_size, output_size):
    super(SimpleModel, self).__init__()
    self.layer1 = torch.nn.Linear(input_size, 100, bias=True)
    self.layer2 = torch.nn.Linear(100, output_size, bias=True)
    self.activation = torch.nn.ReLU()

  def forward(self, X):
    y = self.layer1(X)
    y = self.activation(y)
    y = self.layer2(y)
    y = self.activation(y)
    return y

model = SimpleModel(28*28, 10)

In [6]:
train_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

Training loop

In [18]:
optim = torch.optim.SGD(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
n_epochs = 2

for epoch_i in range(n_epochs):
  print(f'Epoch {epoch_i+1}/{n_epochs}')
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optim.step()
    optim.zero_grad()

Epoch 1/2


100%|██████████| 469/469 [00:02<00:00, 209.65it/s]


Epoch 2/2


100%|██████████| 469/469 [00:02<00:00, 207.69it/s]


Now let's test to see how good our model has become!

In [8]:
model.eval()
test_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=True)

(test_X, test_y) = next(iter(test_dataloader))
test_X = test_X.view(-1, 28*28)

output = model(test_X)
predictions = torch.argmax(output, dim=1)
print(predictions)
print(test_y)
corrects = torch.where(predictions == test_y, 1, 0)
print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')

tensor([6, 4, 4, 9, 1, 2, 9, 2, 5, 3, 2, 5, 0, 4, 2, 1, 2, 4, 1, 4, 9, 0, 2, 0,
        4, 6, 3, 9, 0, 8, 2, 8, 2, 8, 1, 1, 3, 9, 3, 1, 5, 9, 4, 8, 3, 9, 8, 4,
        8, 9, 4, 2, 3, 9, 9, 9, 6, 4, 9, 5, 3, 9, 0, 6, 9, 3, 1, 9, 1, 6, 5, 4,
        1, 4, 4, 9, 5, 3, 0, 2, 5, 4, 5, 3, 3, 3, 6, 8, 0, 3, 9, 3, 2, 9, 8, 9,
        0, 4, 6, 6, 4, 9, 6, 6, 9, 9, 9, 9, 4, 0, 5, 1, 5, 4, 9, 0, 9, 9, 3, 9,
        9, 0, 4, 1, 2, 6, 4, 6])
tensor([6, 7, 9, 9, 1, 2, 9, 2, 5, 3, 2, 5, 0, 4, 2, 9, 2, 4, 1, 4, 3, 0, 2, 0,
        8, 6, 3, 7, 0, 8, 2, 8, 2, 8, 1, 1, 3, 9, 3, 4, 5, 7, 4, 8, 3, 9, 9, 4,
        8, 7, 4, 2, 3, 4, 4, 9, 6, 4, 9, 5, 3, 7, 0, 6, 7, 3, 7, 9, 1, 2, 5, 8,
        1, 4, 4, 9, 5, 8, 0, 2, 5, 4, 5, 3, 3, 3, 6, 8, 0, 3, 7, 5, 2, 9, 8, 9,
        0, 4, 0, 6, 4, 9, 6, 6, 8, 7, 7, 9, 4, 0, 5, 1, 5, 4, 7, 0, 7, 9, 3, 2,
        9, 0, 4, 8, 2, 6, 4, 6])
Number of correct predictions: 100/128


# Putting everything together: Validation/Training Loop

In [ ]:
def train_loop():
  model.train()
  corrects = []
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optim.step()
    optim.zero_grad()

    predictions_i = torch.argmax(output, dim=1)
    corrects_i = torch.where(predictions_i == y, 1, 0)
    corrects.extend(corrects_i)
  print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')
  print(f'Accuracy on training dataset: {sum(corrects)/len(corrects)*100:.3f}%')

def test_loop():
  model.eval()
  corrects = []
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)

    predictions_i = torch.argmax(output, dim=1)
    corrects_i = torch.where(predictions_i == y, 1, 0)
    corrects.extend(corrects_i)
  print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')
  print(f'Accuracy on test dataset: {sum(corrects)/len(corrects)*100:.3f}%')

In [ ]:
train_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=True)

optim = torch.optim.SGD(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()
n_epochs = 100

print('Testing model before training')
test_loop()
for epoch_i in range(n_epochs):
  print(f'\nEpoch {epoch_i+1}/{n_epochs}')
  print('Training model...')
  train_loop()
  print('Testing model...')
  test_loop()

Testing model before training


100%|██████████| 469/469 [00:02<00:00, 224.42it/s]


Number of correct predictions: 49771/60000
Accuracy on test dataset: 82.952%
Test loss: 0.4189767837524414

Epoch 1/100
Training model...


100%|██████████| 469/469 [00:02<00:00, 176.66it/s]


Number of correct predictions: 49774/60000
Accuracy on training dataset: 82.957%
Testing model...


...

Number of correct predictions: 49850/60000
Accuracy on training dataset: 83.083%
Testing model...


100%|██████████| 469/469 [00:01<00:00, 255.96it/s]


Number of correct predictions: 49900/60000
Accuracy on test dataset: 83.167%
Test loss: 0.3951537311077118

Epoch 3/100
Training model...


 39%|███▉      | 183/469 [00:00<00:01, 184.75it/s]


KeyboardInterrupt: 